# Extending Datasets

This tutorial shows how to create a custom dataset by extending [`BaseDataset`](https://instadeepai.github.io/alf/api/alf_core/dataset/base_dataset/). We'll implement a simple CSV dataset loader as an example.

## Setup

Run the cell below to install ALF and this tutorial's dependencies — **no repository clone required**, so it works in a fresh environment.

- Already set up a dev environment from a clone (`uv sync`)? You can **skip the install cell**.

For all installation options, see the [Installation Guide](https://instadeepai.github.io/alf/installation.html).

In [ ]:
# Install ALF + this tutorial's dependencies — no clone needed.
# (Skip this cell if you are already running from a cloned repo via `uv sync`.)
# TODO(pypi): once ALF is published to PyPI, replace the git install below with:
#   %pip install alf_core pandas
%pip install "git+https://github.com/instadeepai/alf.git#subdirectory=core" pandas

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
from alf_core.dataclasses import Candidate, LabelledCandidates, Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType

## 2. Define Custom Dataset

To create a custom dataset, extend `BaseDataset` and implement the required method. Here is the minimal class template:

```python
class MyDataset(BaseDataset):
    def load_dataset(self) -> LabelledCandidates:
        """Required: load your data and return it as a LabelledCandidates object."""
        raise NotImplementedError

    def set_metadata(self) -> None:
        """Optional: store any dataset-specific metadata in self.metadata."""
        pass
```

`load_dataset()` must return a [`LabelledCandidates`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/labelled_candidates/) object — a list of [`Candidate`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) objects paired with a numpy array of labels. Each `Candidate` should be created with the appropriate [`Modality`](https://instadeepai.github.io/alf/api/alf_core/dataclasses/candidate/) for your data type (e.g. `TABULAR`, `SEQUENCE`, `MOLECULE`).

Below is a full implementation using a CSV file as the data source:

In [ ]:
class CSVDataset(BaseDataset):
    """Custom dataset that loads data from a CSV file."""

    def __init__(
        self, config: BaseDatasetConfig, csv_path: str, feature_cols: list[str], label_col: str
    ):
        super().__init__(config)
        self.csv_path = csv_path
        self.feature_cols = feature_cols
        self.label_col = label_col

    def load_dataset(self) -> LabelledCandidates:
        """Load dataset from CSV file."""
        # Read CSV
        df = pd.read_csv(self.csv_path)

        # Extract features and labels
        features = df[self.feature_cols].values
        labels = df[self.label_col].values

        # Create Candidate objects
        candidates = [
            Candidate(
                data=row,
                modality=self.config.modality,
                features={"raw": row},  # Store features if needed
            )
            for row in features
        ]

        return LabelledCandidates(candidates=candidates, labels=labels)

    def set_metadata(self) -> None:
        """Optionally set dataset-specific metadata."""
        self.metadata = {
            "source": self.csv_path,
            "num_features": len(self.feature_cols),
            "feature_names": self.feature_cols,
        }

## 3. Configuration

`BaseDatasetConfig` controls how your dataset is split and accessed. All fields are validated — `train_ratio + test_ratio` must be ≤ 1.

| Field | Type | Description |
|---|---|---|
| `name` | `str` | Identifier for the dataset |
| `modality` | `Modality` | Data type: `TABULAR`, `SEQUENCE`, `MOLECULE`, etc. |
| `seed` | `int` | Random seed for reproducible splits |
| `train_ratio` | `float` [0–1] | Fraction of data used for training |
| `validation_frac` | `float` [0–1] | Fraction of the **training** data held out for validation |
| `test_ratio` | `float` [0–1] | Fraction of data used for testing |
| `problem_type` | `ProblemType` | `REGRESSION`, `BINARY`, or `MULTICLASS` — determines metrics and split strategy |
| `split_type` | `"random"` \| `"low_vs_high"` \| `"stratified"` | How to partition the data. `"stratified"` requires a classification `problem_type` and preserves class balance across splits |
| `max_candidate_pool` | `int \| None` | Optional cap on candidate pool size; remaining data after train/test forms the pool |

After calling `dataset.setup()`, splits are accessible via:
- `dataset.train_dataset` — training split (`LabelledCandidates`)
- `dataset.validation_dataset` — validation split (`LabelledCandidates`)
- `dataset.test_dataset` — test split (`LabelledCandidates`)
- `dataset.candidate_pool` — unlabelled pool for active learning (`LabelledCandidates`)
- `dataset.num_classes` — number of distinct classes (1 for regression)

Use `dataset.query(candidates)` to retrieve ground-truth labels for a list of candidates (for offline evaluation).

In [ ]:
# Create configuration
config = BaseDatasetConfig(
    name="my_csv_dataset",
    modality=Modality.TABULAR,
    seed=42,
    train_ratio=0.6,
    validation_frac=0.2,  # 20% of training data
    test_ratio=0.2,
    split_type="random",
    problem_type=ProblemType.REGRESSION,
)

## 4. Usage Example

In [ ]:
# Create sample CSV file for demonstration
sample_data = pd.DataFrame({
    "feature1": np.random.randn(100),
    "feature2": np.random.randn(100),
    "target": np.random.rand(100),
})
sample_data.to_csv("/tmp/sample_data.csv", index=False)

# Initialise dataset
dataset = CSVDataset(
    config=config,
    csv_path="/tmp/sample_data.csv",
    feature_cols=["feature1", "feature2"],
    label_col="target",
)

# Setup: loads data and creates splits
dataset.setup()

# Access splits
print(f"Training samples: {len(dataset.train_dataset)}")
print(f"Validation samples: {len(dataset.validation_dataset)}")
print(f"Test samples: {len(dataset.test_dataset)}")
print(f"Candidate pool: {len(dataset.candidate_pool)}")


# Query labels for specific candidates
sample_candidates = dataset.candidate_pool.candidates[:5]
labelled = dataset.query(sample_candidates)
print(f"\nQueried {len(labelled)} candidates with labels")